In [ ]:
# 1. Trích xuất dataset địa danh chuẩn xác từ Parquet Shards
import io
import json
import re
import urllib.request
from collections import Counter
from pathlib import Path
from datasets import load_dataset
from PIL import Image

REPO_ID = "Dangindev/viet-cultural-vqa"
OUTPUT_DIR = Path("dataset_landmarks")
TRAIN_DIR = OUTPUT_DIR / "train"
VAL_DIR = OUTPUT_DIR / "val"
TRAIN_DIR.mkdir(parents=True, exist_ok=True)
VAL_DIR.mkdir(parents=True, exist_ok=True)

LANDMARK_MAPPINGS = {
    "chua_mot_cot": ["mot cot", "chùa một cột", "chua mot cot"],
    "ho_hoan_kiem": ["hoan kiem", "ho guom", "hồ gươm", "the huc", "tháp rùa", "ngoc son"],
    "van_mieu": ["van mieu", "văn miếu", "quoc tu giam"],
    "chua_tran_quoc": ["tran quoc", "trấn quốc"],
    "cau_rong": ["cau rong", "cầu rồng"],
    "cau_vang": ["cau vang", "cầu vàng"],
    "chua_linh_ung": ["linh ung", "linh ứng"],
    "ngu_hanh_son": ["ngu hanh son", "ngũ hành sơn"],
    "chua_cau_hoi_an": ["chua cau", "chùa cầu", "nhat ban hoi an", "cầu nhật bản", "hoi an ancient town"],
    "thanh_dia_my_son": ["my son", "mỹ sơn"],
    "ngo_mon_hue": ["ngo mon", "ngọ môn", "kinh thanh hue", "hoang thanh hue"],
    "chua_thien_mu": ["thien mu", "thiên mụ"],
    "lang_khai_dinh": ["khai dinh", "khải định"],
    "lang_tu_duc": ["tu duc", "tự đức"],
    "cho_ben_thanh": ["ben thanh", "bến thành"],
    "dinh_doc_lap": ["doc lap", "độc lập"],
    "nha_tho_duc_ba": ["duc ba", "đức bà"],
    "cau_long_bien": ["long bien", "long biên"],
}

def normalize_text(text: str) -> str:
    text = text.lower()
    patterns = {
        "[àáảãạăằắẳẵặâầấẩẫậ]": "a", "[èéẻẽẹêềếểễệ]": "e", "[ìíỉĩị]": "i",
        "[òóỏõọôồốổỗộơờớởỡợ]": "o", "[ùúủũụưừứửữự]": "u", "[ỳýỷỹỵ]": "y", "đ": "d",
    }
    for pat, rep in patterns.items():
        text = re.sub(pat, rep, text)
    return re.sub(r"[^a-z0-9]", " ", text)

def match_landmark(keyword: str) -> str | None:
    norm_kw = normalize_text(keyword)
    for landmark_id, aliases in LANDMARK_MAPPINGS.items():
        for alias in aliases:
            if normalize_text(alias) in norm_kw:
                return landmark_id
    return None

api_url = f"https://datasets-server.huggingface.co/parquet?dataset={REPO_ID}"
req = urllib.request.Request(api_url, headers={"User-Agent": "Mozilla/5.0"})
with urllib.request.urlopen(req, timeout=30) as resp:
    parquet_data = json.loads(resp.read().decode("utf-8"))
files = parquet_data.get("parquet_files", [])
train_urls = [f["url"] for f in files if f.get("split") == "train"]
val_urls = [f["url"] for f in files if f.get("split") in ("val", "validation")]

def extract(urls, split_name, target_dir, max_per_class=50):
    ds = load_dataset("parquet", data_files={split_name: urls}, split=split_name, streaming=True)
    saved = Counter()
    for item in ds:
        lm_id = match_landmark(item.get("keyword", ""))
        if lm_id and saved[lm_id] < max_per_class:
            folder = target_dir / lm_id
            folder.mkdir(parents=True, exist_ok=True)
            raw = item.get("image")
            if raw:
                img = Image.open(io.BytesIO(raw["bytes"])) if isinstance(raw, dict) and "bytes" in raw else raw
                if hasattr(img, "save"):
                    img.convert("RGB").save(folder / f"{split_name}_{saved[lm_id]:03d}.jpg", quality=90)
                    saved[lm_id] += 1
                    print(f"  📸 [{split_name}] {lm_id}: {saved[lm_id]}/{max_per_class} ảnh", end="\r")
        if all(saved[k] >= max_per_class for k in LANDMARK_MAPPINGS):
            break
    print(f"\n✅ Hoàn thành {split_name}!")

extract(train_urls, "train", TRAIN_DIR, max_per_class=50)
extract(val_urls or [train_urls[-1]], "validation", VAL_DIR, max_per_class=15)
